# Sistemas de Equações Lineares — notebook resolvido

Notebook com os códigos do material executados e os exercícios resolvidos.

In [1]:
import numpy as np
import scipy.linalg as sla

## Funções do material

In [2]:
def sist_lin_diagonal(A, b):
    return b / A.diagonal()

def sist_lin_tri_sup(A, b):
    n = len(b)
    x = np.empty(n, dtype=float)
    x[-1] = b[-1] / A[-1, -1]
    for i in range(n - 2, -1, -1):
        x[i] = (b[i] - np.sum(A[i, i+1:] * x[i+1:])) / A[i, i]
    return x

def sist_lin_tri_inf(A, b):
    n = len(b)
    x = np.empty(n, dtype=float)
    x[0] = b[0] / A[0, 0]
    for i in range(1, n):
        x[i] = (b[i] - np.sum(A[i, :i] * x[:i])) / A[i, i]
    return x

def classificar_sistema(A, b):
    rank_A = np.linalg.matrix_rank(A)
    rank_Ae = np.linalg.matrix_rank(np.c_[A, b])
    n = A.shape[1]
    if rank_A == rank_Ae == n:
        return "possível e determinado"
    if rank_A == rank_Ae < n:
        return "possível e indeterminado"
    return "impossível"

def resolver_sistema_trivial(A, b):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)

    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("A matriz de coeficientes precisa ser quadrada.")

    if np.allclose(A, np.diag(np.diag(A))):
        if np.any(np.isclose(np.diag(A), 0)):
            raise ValueError("Matriz diagonal com elemento nulo na diagonal principal.")
        return b / np.diag(A)

    if np.allclose(A, np.triu(A)):
        if np.any(np.isclose(np.diag(A), 0)):
            raise ValueError("Matriz triangular superior com pivô nulo na diagonal principal.")
        return sist_lin_tri_sup(A, b)

    if np.allclose(A, np.tril(A)):
        if np.any(np.isclose(np.diag(A), 0)):
            raise ValueError("Matriz triangular inferior com pivô nulo na diagonal principal.")
        return sist_lin_tri_inf(A, b)

    raise ValueError("A matriz não é diagonal, triangular superior nem triangular inferior.")

## Exemplo: sistema diagonal

In [3]:
A = np.diag([1, 4, 2])
b = np.array([4, 2, 2], dtype=float)
sist_lin_diagonal(A, b)

array([4. , 0.5, 1. ])

## Exercício 1.1 — classificação

In [4]:
A1 = np.array([[1, 2, 3],
               [4, 5, 6],
               [7, 8, 9]], dtype=float)
b1 = np.array([1, 1, 1], dtype=float)

print("Sistema (a):")
print("rank(A)  =", np.linalg.matrix_rank(A1))
print("rank(Ae) =", np.linalg.matrix_rank(np.c_[A1, b1]))
print("Classificação:", classificar_sistema(A1, b1))

Sistema (a):
rank(A)  = 2
rank(Ae) = 2
Classificação: possível e indeterminado


In [5]:
A2 = np.array([[2, 3],
               [-4, -6]], dtype=float)
b2 = np.array([10, -10], dtype=float)

print("Sistema (b):")
print("rank(A)  =", np.linalg.matrix_rank(A2))
print("rank(Ae) =", np.linalg.matrix_rank(np.c_[A2, b2]))
print("Classificação:", classificar_sistema(A2, b2))

Sistema (b):
rank(A)  = 1
rank(Ae) = 2
Classificação: impossível


Resposta do exercício 1.1:

- Sistema (a): possível e indeterminado.
- Sistema (b): impossível.

## Exemplo 3.2 — triangular superior, passo a passo

In [6]:
A = np.array([[3, 4, -5, 1],
              [0, 1,  1, -2],
              [0, 0,  4, -5],
              [0, 0,  0,  2]], dtype=float)
b = np.array([-10, -1, 3, 2], dtype=float)

x = np.empty(len(b), dtype=float)

x[3] = b[3] / A[3, 3]
x[2] = (b[2] - np.sum(A[2, 3:] * x[3:])) / A[2, 2]
x[1] = (b[1] - np.sum(A[1, 2:] * x[2:])) / A[1, 1]
x[0] = (b[0] - np.sum(A[0, 1:] * x[1:])) / A[0, 0]

print("x4 =", x[3])
print("x3 =", x[2])
print("x2 =", x[1])
print("x1 =", x[0])
print("Solução:", x)
print("Verificação:", A.dot(x))

x4 = 1.0
x3 = 2.0
x2 = -1.0
x1 = 1.0
Solução: [ 1. -1.  2.  1.]
Verificação: [-10.  -1.   3.   2.]


In [7]:
print("Função:", sist_lin_tri_sup(A, b))

Função: [ 1. -1.  2.  1.]


## Exercício 1.2 — passos explícitos do triangular superior

A sequência usada acima corresponde exatamente à substituição retroativa:
1. resolve a última equação;
2. substitui o valor obtido na penúltima;
3. repete até chegar à primeira equação.

## Exemplo 3.3 — triangular inferior, passo a passo

In [8]:
A = np.array([[3, 0, 0, 0],
              [2, 1, 0, 0],
              [1, 0, 1, 0],
              [1, 1, 1, 1]], dtype=float)
b = np.array([4, 2, 4, 2], dtype=float)

x = np.empty(len(b), dtype=float)

x[0] = b[0] / A[0, 0]
x[1] = (b[1] - np.sum(A[1, :1] * x[:1])) / A[1, 1]
x[2] = (b[2] - np.sum(A[2, :2] * x[:2])) / A[2, 2]
x[3] = (b[3] - np.sum(A[3, :3] * x[:3])) / A[3, 3]

print("x1 =", x[0])
print("x2 =", x[1])
print("x3 =", x[2])
print("x4 =", x[3])
print("Solução:", x)
print("Verificação:", A.dot(x))

x1 = 1.3333333333333333
x2 = -0.6666666666666665
x3 = 2.666666666666667
x4 = -1.333333333333334
Solução: [ 1.33333333 -0.66666667  2.66666667 -1.33333333]
Verificação: [4. 2. 4. 2.]


In [9]:
print("Função:", sist_lin_tri_inf(A, b))

Função: [ 1.33333333 -0.66666667  2.66666667 -1.33333333]


## Exercício 1.3 — passos explícitos do triangular inferior

Aqui a ideia é a substituição progressiva:
1. resolve a primeira equação;
2. usa o resultado na segunda;
3. continua até a última equação.

## Exercício 1.4 — função geral

In [10]:
A_diag = np.diag([1, 4, 2])
b_diag = np.array([4, 2, 2], dtype=float)

A_sup = np.array([[3, 4, -5, 1],
                  [0, 1,  1, -2],
                  [0, 0,  4, -5],
                  [0, 0,  0,  2]], dtype=float)
b_sup = np.array([-10, -1, 3, 2], dtype=float)

A_inf = np.array([[3, 0, 0, 0],
                  [2, 1, 0, 0],
                  [1, 0, 1, 0],
                  [1, 1, 1, 1]], dtype=float)
b_inf = np.array([4, 2, 4, 2], dtype=float)

print("Diagonal :", resolver_sistema_trivial(A_diag, b_diag))
print("Superior :", resolver_sistema_trivial(A_sup, b_sup))
print("Inferior :", resolver_sistema_trivial(A_inf, b_inf))

Diagonal : [4.  0.5 1. ]
Superior : [ 1. -1.  2.  1.]
Inferior : [ 1.33333333 -0.66666667  2.66666667 -1.33333333]


## Resumo final

- Os códigos do material foram colocados e executados.
- O exercício 1.1 foi classificado.
- Os exercícios 1.2 e 1.3 foram mostrados com cada passo da solução.
- O exercício 1.4 foi implementado em uma função que detecta o tipo da matriz e resolve o sistema.